# Protocol Amendment A1 — Official-Route Recovery and Development-Only Substitution Rules

This prospective amendment responds to the sealed Stage 11B result: none of the five preassigned breast-ultrasound development domains became repair-ready under the original transport rules, and no source-recoverability performance failure was observed. A1 changes **only** development acquisition governance.

The original five development datasets retain first priority. A1 permits current official APIs, provider landing-page downloads, official clients/tools, publisher or author repositories, and a checksum-audited manual upload from an official page. If pre-model acquisition and governance gates still leave open development slots, a frozen reserve pool may fill them in fixed order. Selection may never use embedding, source-gate, transfer, calibration, DDO2, or blind-test outcomes.

Stage 9's 21-edge library and model specification remain unchanged. The three locked-blind datasets and their 13 candidate blind edges remain permanently locked and are neither probed nor accessed here. This notebook downloads no image, label, archive, or metadata; fits no model; and performs no validation. It only seals the amendment and an exact handoff to an amended Stage 11C.

In [10]:
# @title A1-0. Mount Drive, verify sealed lineage, and initialise an immutable amendment workspace
import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    pass

PROJECT_ROOT = Path(os.environ.get("CDO_PROJECT_ROOT", "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability"))
CODE_ROOT = PROJECT_ROOT / "05_Code" / "Cross_Modal"
CROSS_MODAL_ROOT = PROJECT_ROOT / "06_Data_Records" / "Cross_Modal"
STAGE9_ROOT = CROSS_MODAL_ROOT / "Stage9_Hierarchical_DDO2_Specification_Freeze_v0.1"
STAGE10_ROOT = CROSS_MODAL_ROOT / "Stage10_Preassigned_Development_And_Locked_Blind_Registry_v0.1"
STAGE11B_ROOT = CROSS_MODAL_ROOT / "Stage11B_Protocol_Conformant_Reacquisition_And_Patient_Grouping_Repair_v0.1"
A1_ROOT = CROSS_MODAL_ROOT / "Protocol_Amendment_A1_Official_Route_Recovery_And_Development_Only_Substitution_Rules_v0.1"

PROTOCOL_ROOT = A1_ROOT / "00_Protocol"
LINEAGE_ROOT = A1_ROOT / "01_Lineage"
RECOVERY_ROOT = A1_ROOT / "02_Official_Route_Recovery"
RESERVE_ROOT = A1_ROOT / "03_Development_Reserve"
FIREWALL_ROOT = A1_ROOT / "04_Firewall"
RESULT_ROOT = A1_ROOT / "05_Results"
for directory in [CODE_ROOT, PROTOCOL_ROOT, LINEAGE_ROOT, RECOVERY_ROOT, RESERVE_ROOT, FIREWALL_ROOT, RESULT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = CODE_ROOT / "CrossModal_Protocol_Amendment_A1_Official_Route_Recovery_And_Development_Only_Substitution_Rules_v0.1.ipynb"
STAGE9_FINAL_PATH = STAGE9_ROOT / "05_Results" / "Stage9_Hierarchical_DDO2_Specification_Freeze_Complete_v0.1.json"
STAGE10_FINAL_PATH = STAGE10_ROOT / "05_Results" / "Stage10_Preassigned_Role_Registry_Complete_v0.1.json"
STAGE10_ROLE_PATH = STAGE10_ROOT / "01_Role_Registry" / "Stage10_Frozen_Dataset_Acquisition_Role_Registry_v0.1.csv"
STAGE11B_FINAL_PATH = STAGE11B_ROOT / "05_Results" / "Stage11B_Protocol_Conformant_Repair_Complete_v0.1.json"

PROTOCOL_SEAL_PATH = PROTOCOL_ROOT / "Protocol_Amendment_A1_Seal_v0.1.json"
PARENT_COMMITMENT_PATH = LINEAGE_ROOT / "A1_Parent_Input_Commitment_v0.1.csv"
ROUTE_REGISTRY_PATH = RECOVERY_ROOT / "A1_Official_Route_Recovery_Registry_v0.1.csv"
MANUAL_RECEIPT_SCHEMA_PATH = RECOVERY_ROOT / "A1_Manual_Official_Download_Receipt_Schema_v0.1.csv"
RESERVE_POOL_PATH = RESERVE_ROOT / "A1_Frozen_Development_Reserve_Pool_v0.1.csv"
EXCLUSION_REGISTRY_PATH = RESERVE_ROOT / "A1_Explicit_Exclusion_Registry_v0.1.csv"
ACTIVATION_POLICY_PATH = RESERVE_ROOT / "A1_Outcome_Free_Activation_Policy_v0.1.json"
BLIND_CONTINUITY_PATH = FIREWALL_ROOT / "A1_Locked_Blind_Continuity_Registry_v0.1.csv"
VALIDITY_PATH = FIREWALL_ROOT / "A1_Independent_Validity_Checks_v0.1.csv"
REPORT_PATH = RESULT_ROOT / "Protocol_Amendment_A1_Report_v0.1.md"
FIGURE_PATH = RESULT_ROOT / "A1_Development_Capacity_Logic_v0.1.png"
OUTPUT_MANIFEST_PATH = RESULT_ROOT / "A1_Output_Integrity_Manifest_v0.1.csv"
RUNTIME_STATE_PATH = RESULT_ROOT / "A1_Runtime_State_v0.1.json"
FINAL_RECORD_PATH = RESULT_ROOT / "Protocol_Amendment_A1_Complete_v0.1.json"

EXPECTED_STAGE9_FINAL_HASH = "2568a9fdaff83ab74938655d65d25cd9d50f8b28bafbc44e856098932f6c6648"
EXPECTED_STAGE10_FINAL_HASH = "6438434cb41607ad97b1b4a2fab07b143969ce7f551b87b5c2a23afbe67ccccf"
EXPECTED_STAGE10_PROTOCOL_HASH = "305c715cd95c72b7e54cfd241891080ddd6838e4428c6bd5acdf7b4264c96f31"
EXPECTED_ROLE_REGISTRY_HASH = "cc0b56178eecee123d8c6fa99824018b18aae79679c6620edc7e5ebcc9018bc3"
EXPECTED_STAGE11B_FINAL_HASH = "d77c944ee88674696a069831caadd2131bc6dd6a7dad996775967694088664e3"
EXPECTED_STAGE11B_PROTOCOL_HASH = "f09e2b0d9505bded7329d9b380c098542cc9450b6e16d695295228da18ac67eb"
MAXIMUM_NEW_A1_BYTES = 32 * 1024 * 1024

ORIGINAL_IDS = ["BUS_BRA_2024", "BUSI_WHU_2025_V3", "BREAST_LESIONS_USG_2024", "BUS_UCLM_2025_V3", "RODRIGUES_BUI_2017"]
LOCKED_BLIND_IDS = ["BUSI_CAIRO_2019", "OASBUD_2017", "DERM7PT_2019"]

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def sha256_json(payload):
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def canonical_csv_text(frame):
    return frame.to_csv(index=False, lineterminator="\n", float_format="%.12g")

def write_immutable_text(path, text):
    path = Path(path)
    if path.is_file():
        assert path.read_text(encoding="utf-8") == text, f"Existing immutable text differs: {path}"
    else:
        path.write_text(text, encoding="utf-8")

def write_immutable_csv(path, frame):
    write_immutable_text(path, canonical_csv_text(frame))

def write_immutable_json(path, payload):
    write_immutable_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")

def atomic_json(path, payload):
    temporary = Path(str(path) + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
    os.replace(temporary, path)

def verify_self_hashed_json(path, expected_hash, hash_field="final_record_sha256"):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    claim = payload.get(hash_field)
    assert claim == expected_hash, f"Unexpected frozen claim: {path}"
    without_claim = dict(payload)
    without_claim.pop(hash_field)
    assert sha256_json(without_claim) == claim, f"Self-hash mismatch: {path}"
    return payload

def normalised_notebook_source_sha256(path):
    notebook = json.loads(Path(path).read_text(encoding="utf-8"))
    payload = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") in {"code", "markdown"}:
            source = cell.get("source", [])
            source = "".join(source) if isinstance(source, list) else str(source)
            payload.append({"cell_type": cell["cell_type"], "source": source.replace("\r\n", "\n")})
    return sha256_json(payload)

required = [NOTEBOOK_PATH, STAGE9_FINAL_PATH, STAGE10_FINAL_PATH, STAGE10_ROLE_PATH, STAGE11B_FINAL_PATH]
missing_required = [path for path in required if not path.is_file()]
assert not missing_required, "A sealed parent is missing; do not substitute a similarly named file. Missing:\n" + "\n".join(map(str, missing_required))
stage9_final = verify_self_hashed_json(STAGE9_FINAL_PATH, EXPECTED_STAGE9_FINAL_HASH)
stage10_final = verify_self_hashed_json(STAGE10_FINAL_PATH, EXPECTED_STAGE10_FINAL_HASH)
stage11b_final = verify_self_hashed_json(STAGE11B_FINAL_PATH, EXPECTED_STAGE11B_FINAL_HASH)
assert sha256_file(STAGE10_ROLE_PATH) == EXPECTED_ROLE_REGISTRY_HASH
assert stage10_final["stage10_protocol_seal_sha256"] == EXPECTED_STAGE10_PROTOCOL_HASH
assert stage11b_final["stage11b_protocol_seal_sha256"] == EXPECTED_STAGE11B_PROTOCOL_HASH

role_registry = pd.read_csv(STAGE10_ROLE_PATH)
development = role_registry[role_registry["role_frozen_before_label_access"].eq("DEVELOPMENT_EXTENSION")].copy()
locked_blind = role_registry[role_registry["role_frozen_before_label_access"].eq("LOCKED_BLIND_TEST")].copy()
assert development["dataset_id"].tolist() == ORIGINAL_IDS
assert locked_blind["dataset_id"].tolist() == LOCKED_BLIND_IDS

REPLAY_MODE = FINAL_RECORD_PATH.is_file()
notebook_source_hash = normalised_notebook_source_sha256(NOTEBOOK_PATH)
parent_commitment = pd.DataFrame([
    {"role": "STAGE9_FINAL", "relative_path": str(STAGE9_FINAL_PATH.relative_to(PROJECT_ROOT)), "size_bytes": STAGE9_FINAL_PATH.stat().st_size, "sha256": sha256_file(STAGE9_FINAL_PATH), "self_hash_claim": EXPECTED_STAGE9_FINAL_HASH},
    {"role": "STAGE10_FINAL", "relative_path": str(STAGE10_FINAL_PATH.relative_to(PROJECT_ROOT)), "size_bytes": STAGE10_FINAL_PATH.stat().st_size, "sha256": sha256_file(STAGE10_FINAL_PATH), "self_hash_claim": EXPECTED_STAGE10_FINAL_HASH},
    {"role": "STAGE10_ROLE_REGISTRY", "relative_path": str(STAGE10_ROLE_PATH.relative_to(PROJECT_ROOT)), "size_bytes": STAGE10_ROLE_PATH.stat().st_size, "sha256": sha256_file(STAGE10_ROLE_PATH), "self_hash_claim": EXPECTED_ROLE_REGISTRY_HASH},
    {"role": "STAGE11B_FINAL", "relative_path": str(STAGE11B_FINAL_PATH.relative_to(PROJECT_ROOT)), "size_bytes": STAGE11B_FINAL_PATH.stat().st_size, "sha256": sha256_file(STAGE11B_FINAL_PATH), "self_hash_claim": EXPECTED_STAGE11B_FINAL_HASH},
])
write_immutable_csv(PARENT_COMMITMENT_PATH, parent_commitment)

runtime_state = {
    "stage": "ProtocolAmendmentA1", "replay_mode": REPLAY_MODE, "parents_verified": True,
    "external_requests_made": False, "images_or_labels_accessed": False,
    "locked_blind_assets_touched": False, "models_fitted": False, "last_updated_utc": utc_now(),
}
if not REPLAY_MODE:
    atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("Stage 9 / 10 / 11B final records verified:", EXPECTED_STAGE9_FINAL_HASH, "/", EXPECTED_STAGE10_FINAL_HASH, "/", EXPECTED_STAGE11B_FINAL_HASH)
print("Original development / locked-blind datasets:", len(development), "/", len(locked_blind))
print("Replay mode / external request made:", REPLAY_MODE, "/", False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Stage 9 / 10 / 11B final records verified: 2568a9fdaff83ab74938655d65d25cd9d50f8b28bafbc44e856098932f6c6648 / 6438434cb41607ad97b1b4a2fab07b143969ce7f551b87b5c2a23afbe67ccccf / d77c944ee88674696a069831caadd2131bc6dd6a7dad996775967694088664e3
Original development / locked-blind datasets: 5 / 3
Replay mode / external request made: True / False


In [11]:
# @title A1-1. Freeze the official-route recovery hierarchy for the original five development datasets
route_rows = [
    {
        "dataset_id": "BUS_BRA_2024", "original_priority": 1, "original_role_retained": "DEVELOPMENT_EXTENSION",
        "official_landing_url": "https://zenodo.org/records/8231412",
        "permitted_route_order": "ZENODO_CURRENT_RECORD_API|ZENODO_LANDING_DOWNLOAD_ALL|ZENODO_OFFICIAL_FILE_URL|PUBLISHER_AUTHOR_OFFICIAL_REPOSITORY|MANUAL_OFFICIAL_DOWNLOAD_WITH_CHECKSUM_RECEIPT",
        "permitted_hosts": "zenodo.org|files.zenodo.org|github.com/wgomezf/BUS-BRA|raw.githubusercontent.com/wgomezf/BUS-BRA",
        "required_identity_gate": "OFFICIAL_RECORD_VERSION_PLUS_PROVIDER_MD5_OR_SHA256_PLUS_ARCHIVE_FILE_INVENTORY",
        "required_governance_gate": "PATIENT_GROUPING_EXPLICITLY_CORROBORATED; RELEASED_PATHOLOGY_MAPPING; LICENCE; DUPLICATE_AUDIT",
    },
    {
        "dataset_id": "BUSI_WHU_2025_V3", "original_priority": 2, "original_role_retained": "DEVELOPMENT_EXTENSION",
        "official_landing_url": "https://data.mendeley.com/datasets/k6cpmwybk3/3",
        "permitted_route_order": "MENDELEY_LANDING_DOWNLOAD_ALL|MENDELEY_OAUTH_OFFICIAL_API|MENDELEY_OFFICIAL_FILE_URL|MANUAL_OFFICIAL_DOWNLOAD_WITH_CHECKSUM_RECEIPT",
        "permitted_hosts": "data.mendeley.com|api.mendeley.com|api.data.mendeley.com",
        "required_identity_gate": "DOI_VERSION_3_PLUS_OFFICIAL_FILE_MANIFEST_PLUS_SHA256",
        "required_governance_gate": "PATIENT_GROUP_MAP; RELEASED_DIAGNOSIS_MAPPING; LICENCE; DUPLICATE_AUDIT",
    },
    {
        "dataset_id": "BREAST_LESIONS_USG_2024", "original_priority": 3, "original_role_retained": "DEVELOPMENT_EXTENSION",
        "official_landing_url": "https://www.cancerimagingarchive.net/collection/breast-lesions-usg/",
        "permitted_route_order": "TCIA_NBIA_CURRENT_API|TCIA_DATA_RETRIEVER|TCIA_UTILS_OFFICIAL_CLIENT|TCIA_LANDING_MANIFEST|MANUAL_OFFICIAL_NBIA_DOWNLOAD_WITH_MANIFEST_RECEIPT",
        "permitted_hosts": "cancerimagingarchive.net|www.cancerimagingarchive.net|nbia.cancerimagingarchive.net|services.cancerimagingarchive.net",
        "required_identity_gate": "TCIA_COLLECTION_DOI_PLUS_SERIES_UID_MANIFEST_PLUS_DICOM_OR_EXPORT_CHECKSUMS",
        "required_governance_gate": "CLINICAL_TABLE_JOIN; CONFIRMED_DIAGNOSIS_MAPPING; ONE_PATIENT_GROUP; LICENCE; OASBUD_PROVENANCE_CLEARANCE_DEFERRED_TO_BLIND_FIREWALL",
    },
    {
        "dataset_id": "BUS_UCLM_2025_V3", "original_priority": 4, "original_role_retained": "DEVELOPMENT_EXTENSION",
        "official_landing_url": "https://data.mendeley.com/datasets/7fvgj4jsp7/3",
        "permitted_route_order": "MENDELEY_LANDING_DOWNLOAD_ALL|MENDELEY_OAUTH_OFFICIAL_API|MENDELEY_OFFICIAL_FILE_URL|PUBLISHER_AUTHOR_OFFICIAL_REPOSITORY|MANUAL_OFFICIAL_DOWNLOAD_WITH_CHECKSUM_RECEIPT",
        "permitted_hosts": "data.mendeley.com|api.mendeley.com|api.data.mendeley.com|github.com",
        "required_identity_gate": "DOI_VERSION_3_PLUS_OFFICIAL_FILE_MANIFEST_PLUS_SHA256",
        "required_governance_gate": "38_PATIENT_MAP; RELEASED_MASK_COLOUR_TO_DIAGNOSIS_MAPPING; LICENCE; DUPLICATE_AUDIT",
    },
    {
        "dataset_id": "RODRIGUES_BUI_2017", "original_priority": 5, "original_role_retained": "DEVELOPMENT_EXTENSION",
        "official_landing_url": "https://data.mendeley.com/datasets/wmy84gzngw/1",
        "permitted_route_order": "MENDELEY_LANDING_DOWNLOAD_ALL|MENDELEY_OAUTH_OFFICIAL_API|MENDELEY_OFFICIAL_FILE_URL|PUBLISHER_AUTHOR_OFFICIAL_REPOSITORY|MANUAL_OFFICIAL_DOWNLOAD_WITH_CHECKSUM_RECEIPT",
        "permitted_hosts": "data.mendeley.com|api.mendeley.com|api.data.mendeley.com|github.com",
        "required_identity_gate": "DOI_VERSION_1_PLUS_OFFICIAL_FILE_MANIFEST_PLUS_SHA256",
        "required_governance_gate": "PATIENT_GROUP_MAP; RELEASED_CLASS_MAPPING; LICENCE; DERIVED_COPY_AND_CROSS_DATASET_DUPLICATE_AUDIT",
    },
]
route_registry_expected = pd.DataFrame(route_rows)

receipt_fields = [
    ("dataset_id", True, "must be one of the original five or an activated A1 reserve"),
    ("official_landing_url", True, "must equal the frozen official source"),
    ("provider_and_repository_version", True, "DOI/record/version/revision"),
    ("official_filename", True, "filename shown by provider"),
    ("download_method", True, "official landing/API/client/manual official download"),
    ("acquired_utc", True, "UTC timestamp"),
    ("size_bytes", True, "exact downloaded byte count"),
    ("provider_checksum_type", False, "MD5/SHA256 if provider publishes one"),
    ("provider_checksum", False, "published value if available"),
    ("computed_sha256", True, "locally computed full-file SHA256"),
    ("provider_checksum_match", False, "required true when published"),
    ("drive_relative_path", True, "development-only quarantine path"),
    ("source_receipt_notes", True, "redirects, client version, or manual steps"),
]
manual_receipt_schema_expected = pd.DataFrame(receipt_fields, columns=["field", "required", "rule"])

if REPLAY_MODE:
    route_registry = pd.read_csv(ROUTE_REGISTRY_PATH)
    manual_receipt_schema = pd.read_csv(MANUAL_RECEIPT_SCHEMA_PATH)
else:
    route_registry = route_registry_expected.copy()
    manual_receipt_schema = manual_receipt_schema_expected.copy()
    write_immutable_csv(ROUTE_REGISTRY_PATH, route_registry)
    write_immutable_csv(MANUAL_RECEIPT_SCHEMA_PATH, manual_receipt_schema)

assert canonical_csv_text(route_registry) == canonical_csv_text(route_registry_expected)
assert canonical_csv_text(manual_receipt_schema) == canonical_csv_text(manual_receipt_schema_expected)
print("Original datasets with expanded official-route recovery:", len(route_registry))
display(route_registry[["dataset_id", "original_priority", "permitted_route_order", "required_identity_gate"]])

Original datasets with expanded official-route recovery: 5


,dataset_id,original_priority,permitted_route_order,required_identity_gate
0,BUS_BRA_2024,1,ZENODO_CURRENT_RECORD_API|ZENODO_LANDING_DOWNL...,OFFICIAL_RECORD_VERSION_PLUS_PROVIDER_MD5_OR_S...
1,BUSI_WHU_2025_V3,2,MENDELEY_LANDING_DOWNLOAD_ALL|MENDELEY_OAUTH_O...,DOI_VERSION_3_PLUS_OFFICIAL_FILE_MANIFEST_PLUS...
2,BREAST_LESIONS_USG_2024,3,TCIA_NBIA_CURRENT_API|TCIA_DATA_RETRIEVER|TCIA...,TCIA_COLLECTION_DOI_PLUS_SERIES_UID_MANIFEST_P...
3,BUS_UCLM_2025_V3,4,MENDELEY_LANDING_DOWNLOAD_ALL|MENDELEY_OAUTH_O...,DOI_VERSION_3_PLUS_OFFICIAL_FILE_MANIFEST_PLUS...
4,RODRIGUES_BUI_2017,5,MENDELEY_LANDING_DOWNLOAD_ALL|MENDELEY_OAUTH_O...,DOI_VERSION_1_PLUS_OFFICIAL_FILE_MANIFEST_PLUS...


In [12]:
# @title A1-2. Freeze a ranked development-only reserve pool and explicit exclusions before any reserve data access
reserve_rows = [
    {
        "reserve_priority": 1, "dataset_id": "UDIAT_B_2017", "role_if_activated": "DEVELOPMENT_EXTENSION_A1_RESERVE",
        "modality": "breast_ultrasound", "task": "breast_lesion_malignant_vs_benign", "expected_units": 163, "unit_type": "STILL_IMAGE_ONE_REPORTED_MASS_PER_IMAGE",
        "official_primary_record": "https://doi.org/10.1109/JBHI.2017.2731873", "official_access_route": "AUTHOR_OR_DATASET_OWNER_REQUEST_VIA_OFFICIAL_PUBLICATION_OR_AUTHOR_DATABASE_PAGE",
        "licence_status_at_seal": "UNKNOWN_RESEARCH_REQUEST_REQUIRED", "label_status_at_seal": "BENIGN_MALIGNANT_REPORTED_PATHOLOGY_BASIS_MUST_BE_VERIFIED",
        "patient_grouping_status_at_seal": "ONE_MASS_PER_IMAGE_REPORTED_BUT_PATIENT_IDENTITY_MUST_BE_PROVEN",
        "mandatory_pre_activation_gates": "OFFICIAL_ACCESS_AND_RIGHTS|PATHOLOGY_COMPATIBLE_LABELS|PATIENT_OR_LESION_GROUP_MAP|CHECKSUM|NO_DERIVATIVE_OR_CROSS_DATASET_DUPLICATE",
        "status": "CONDITIONAL_RESERVE_NOT_ACCESSED_NOT_ACTIVATED",
    },
    {
        "reserve_priority": 2, "dataset_id": "MICCAI_BUV_2022", "role_if_activated": "DEVELOPMENT_EXTENSION_A1_RESERVE",
        "modality": "breast_ultrasound", "task": "breast_lesion_malignant_vs_benign", "expected_units": 188, "unit_type": "ULTRASOUND_VIDEO",
        "official_primary_record": "https://github.com/jhl-Det/CVA-Net", "official_access_route": "PUBLISHER_AUTHOR_OFFICIAL_GITHUB_LINKED_BAIDU_OR_GOOGLE_DRIVE",
        "licence_status_at_seal": "DATASET_README_NONCOMMERCIAL_RESEARCH_OR_EDUCATION_VERIFY_EXACT_TERMS", "label_status_at_seal": "VIDEO_BENIGN_MALIGNANT_LABELS_PATHOLOGY_BASIS_MUST_BE_VERIFIED",
        "patient_grouping_status_at_seal": "KNOWN_DUPLICATE_AND_SAME_PATIENT_VIDEO_CORRECTIONS_REQUIRE_COMPLETE_PATIENT_MAP",
        "mandatory_pre_activation_gates": "OFFICIAL_AUTHOR_ASSET|EXACT_TERMS|CORRECTED_LABEL_RELEASE|COMPLETE_PATIENT_GROUP_MAP|VIDEO_LEVEL_GROUPING|FROZEN_FRAME_SAMPLING_AND_VIDEO_AGGREGATION|DUPLICATE_CLEARANCE",
        "status": "CONDITIONAL_RESERVE_NOT_ACCESSED_NOT_ACTIVATED",
    },
    {
        "reserve_priority": 3, "dataset_id": "UVBLS200_2025", "role_if_activated": "DEVELOPMENT_EXTENSION_A1_RESERVE",
        "modality": "breast_ultrasound", "task": "breast_lesion_malignant_vs_benign", "expected_units": 200, "unit_type": "ULTRASOUND_VIDEO_10666_FRAMES_REPORTED",
        "official_primary_record": "https://github.com/zzgzzgz/STPF-Net", "official_access_route": "PUBLISHER_AUTHOR_OFFICIAL_REPOSITORY_ONLY",
        "licence_status_at_seal": "UNKNOWN_VERIFY_BEFORE_DOWNLOAD", "label_status_at_seal": "80_BENIGN_120_MALIGNANT_REPORTED_PATHOLOGY_BASIS_MUST_BE_VERIFIED",
        "patient_grouping_status_at_seal": "COLLECTION_REPORTS_528_PATIENTS_BUT VIDEO_TO_PATIENT_MAPPING_MUST_BE VERIFIED",
        "mandatory_pre_activation_gates": "OFFICIAL_AUTHOR_ASSET|LICENCE|PATHOLOGY_COMPATIBLE_LABELS|VIDEO_TO_PATIENT_MAP|FRAME_VIDEO_GROUPING|FROZEN_FRAME_SAMPLING_AND_VIDEO_AGGREGATION|DUPLICATE_CLEARANCE",
        "status": "CONDITIONAL_RESERVE_NOT_ACCESSED_NOT_ACTIVATED",
    },
]
reserve_pool_expected = pd.DataFrame(reserve_rows)

exclusion_rows = [
    {"dataset_id_or_family": "BUS_UC_2023", "source": "https://data.mendeley.com/datasets/3ksd7w7jkx/1", "status": "EXCLUDED_FROM_A1_ACTIVATION", "reason": "images originate from UltrasoundCases and were secondarily reannotated; pathology-compatible released diagnosis, patient identity, and upstream image rights are not established"},
    {"dataset_id_or_family": "BUSC_2023", "source": "https://data.mendeley.com/datasets/vckdnhtw26/1", "status": "EXCLUDED_DERIVATIVE_DUPLICATE", "reason": "described as a resized/annotated derivative of the same 100 benign plus 150 malignant Rodrigues/Mendeley image collection"},
    {"dataset_id_or_family": "BUSI_CAIRO_MIRRORS", "source": "Kaggle, Zenodo reposts, Hugging Face, or other mirrors", "status": "EXCLUDED_LOCKED_BLIND_LEAKAGE", "reason": "BUSI_CAIRO_2019 is permanently locked blind; copies, mirrors, transformed versions, or relabelled releases cannot enter development"},
    {"dataset_id_or_family": "BUS_COT_2026_COMPOSITE", "source": "https://doi.org/10.6084/m9.figshare.30838715", "status": "EXCLUDED_COMPOSITE_PROVENANCE", "reason": "aggregates publications, public datasets, and online case repositories; cannot be treated as one independent domain and may contain original or blind-domain overlap"},
]
exclusion_registry_expected = pd.DataFrame(exclusion_rows)

if REPLAY_MODE:
    reserve_pool = pd.read_csv(RESERVE_POOL_PATH)
    exclusion_registry = pd.read_csv(EXCLUSION_REGISTRY_PATH)
else:
    reserve_pool = reserve_pool_expected.copy()
    exclusion_registry = exclusion_registry_expected.copy()
    write_immutable_csv(RESERVE_POOL_PATH, reserve_pool)
    write_immutable_csv(EXCLUSION_REGISTRY_PATH, exclusion_registry)

assert canonical_csv_text(reserve_pool) == canonical_csv_text(reserve_pool_expected)
assert canonical_csv_text(exclusion_registry) == canonical_csv_text(exclusion_registry_expected)
print("Frozen conditional reserves / explicit exclusions:", len(reserve_pool), "/", len(exclusion_registry))
display(reserve_pool[["reserve_priority", "dataset_id", "expected_units", "status", "mandatory_pre_activation_gates"]])

Frozen conditional reserves / explicit exclusions: 3 / 4


,reserve_priority,dataset_id,expected_units,status,mandatory_pre_activation_gates
0,1,UDIAT_B_2017,163,CONDITIONAL_RESERVE_NOT_ACCESSED_NOT_ACTIVATED,OFFICIAL_ACCESS_AND_RIGHTS|PATHOLOGY_COMPATIBL...
1,2,MICCAI_BUV_2022,188,CONDITIONAL_RESERVE_NOT_ACCESSED_NOT_ACTIVATED,OFFICIAL_AUTHOR_ASSET|EXACT_TERMS|CORRECTED_LA...
2,3,UVBLS200_2025,200,CONDITIONAL_RESERVE_NOT_ACCESSED_NOT_ACTIVATED,OFFICIAL_AUTHOR_ASSET|LICENCE|PATHOLOGY_COMPAT...


In [13]:
# @title A1-3. Freeze outcome-free activation, roster closure, and exact capacity arithmetic
CURRENT_ELIGIBLE_EDGES = 21
MINIMUM_TOTAL_EDGES = 30
ADDITIONAL_EDGES_NEEDED = MINIMUM_TOTAL_EDGES - CURRENT_ELIGIBLE_EDGES
TARGET_DEVELOPMENT_DOMAIN_COUNT = 5

capacity_rows = []
for qualified_domains in range(1, TARGET_DEVELOPMENT_DOMAIN_COUNT + 1):
    maximum_new_edges = qualified_domains * (qualified_domains - 1)
    minimum_recoverable_sources = math.ceil(ADDITIONAL_EDGES_NEEDED / (qualified_domains - 1)) if qualified_domains > 1 else None
    count_gate_possible = bool(qualified_domains > 1 and minimum_recoverable_sources <= qualified_domains)
    capacity_rows.append({
        "qualified_development_domains": qualified_domains,
        "maximum_new_directed_edges_if_all_sources_recoverable": maximum_new_edges,
        "minimum_recoverable_sources_for_nine_new_edges": minimum_recoverable_sources,
        "global_edge_count_gate_possible": count_gate_possible,
    })
capacity_table = pd.DataFrame(capacity_rows)
MINIMUM_QUALIFIED_DOMAINS_FOR_COUNT_GATE = int(capacity_table.loc[capacity_table["global_edge_count_gate_possible"], "qualified_development_domains"].min())
assert MINIMUM_QUALIFIED_DOMAINS_FOR_COUNT_GATE == 4

activation_policy_expected = {
    "stage": "ProtocolAmendmentA1",
    "policy_name": "OUTCOME_FREE_ORIGINAL_FIRST_DEVELOPMENT_RESERVE_ACTIVATION",
    "current_eligible_edges": CURRENT_ELIGIBLE_EDGES,
    "minimum_total_eligible_edges": MINIMUM_TOTAL_EDGES,
    "additional_edges_needed": ADDITIONAL_EDGES_NEEDED,
    "target_development_domain_count": TARGET_DEVELOPMENT_DOMAIN_COUNT,
    "minimum_qualified_domains_with_any_count_gate_possibility": MINIMUM_QUALIFIED_DOMAINS_FOR_COUNT_GATE,
    "edge_formula": "new_eligible_edges = recoverable_sources * (qualified_development_domains - 1)",
    "ordering": [
        "attempt all five originals in frozen Stage10 order through every A1-permitted official route",
        "adjudicate provenance, rights, endpoint, patient grouping, integrity, and development-only duplicate gates before any embedding or model fit",
        "retain every original that passes pre-model gates",
        "if fewer than five domains pass, screen reserves strictly in frozen priority order and activate only enough passing reserves to fill the five-domain roster",
        "seal the final development roster before computing embeddings or fitting any source axis",
    ],
    "activation_inputs_permitted": ["official availability", "licence and use terms", "released label semantics", "patient or lesion grouping", "file integrity", "development-only provenance and duplicate evidence"],
    "activation_inputs_prohibited": ["embedding geometry", "source AUC", "source recoverability", "transfer performance", "calibration", "operating point", "DDO2", "blind image, label, prediction, or outcome"],
    "post_roster_rule": "source-gate failure after roster sealing never activates another reserve and never changes dataset order",
    "insufficient_roster_rule": "if fewer than four acquisition/governance-qualified development domains remain after the frozen reserve pool is exhausted, the Stage9 global edge-count gate is mathematically unreachable and Stage12 remains HOLD",
    "source_fit_rule": "if at least four domains qualify, amended Stage11C may fit the already-frozen source axis for every roster domain only after the roster and all pre-fit gates are sealed",
    "stage12_rule": "requires at least 30 eligible development edges plus every other frozen Stage9 information, class-balance, modality, dataset, and grouped-validation gate; count alone is insufficient",
    "no_further_search_rule": "no unregistered dataset may be inspected or activated without a new prospective amendment sealed before access",
    "blind_boundary": "all three locked-blind roles and thirteen candidate blind edges remain unchanged and inaccessible",
}
activation_policy_expected["policy_sha256"] = sha256_json(activation_policy_expected)

if REPLAY_MODE:
    activation_policy = json.loads(ACTIVATION_POLICY_PATH.read_text(encoding="utf-8"))
else:
    activation_policy = activation_policy_expected.copy()
    write_immutable_json(ACTIVATION_POLICY_PATH, activation_policy)
assert activation_policy == activation_policy_expected

print("Current / required / additionally needed edges:", CURRENT_ELIGIBLE_EDGES, "/", MINIMUM_TOTAL_EDGES, "/", ADDITIONAL_EDGES_NEEDED)
print("Target roster / minimum count-gate-feasible domains:", TARGET_DEVELOPMENT_DOMAIN_COUNT, "/", MINIMUM_QUALIFIED_DOMAINS_FOR_COUNT_GATE)
display(capacity_table)

Current / required / additionally needed edges: 21 / 30 / 9
Target roster / minimum count-gate-feasible domains: 5 / 4


,qualified_development_domains,maximum_new_directed_edges_if_all_sources_recoverable,minimum_recoverable_sources_for_nine_new_edges,global_edge_count_gate_possible
0,1,0,NaN,False
1,2,2,9.0,False
2,3,6,5.0,False
3,4,12,3.0,True
4,5,20,3.0,True


In [14]:
# @title A1-4. Freeze locked-blind continuity and the amended Stage 11C action firewall
blind_continuity_expected = locked_blind[[
    "dataset_id", "modality", "task", "role_frozen_before_label_access", "analysis_use",
    "labels_permitted_before_label_free_prediction_freeze", "eligible_for_model_or_scaler_fit",
    "eligible_for_final_blind_claim", "source_candidacy", "official_landing_url", "persistent_id",
]].copy()
blind_continuity_expected["a1_status"] = "UNCHANGED_PERMANENTLY_LOCKED_BLIND"
blind_continuity_expected["a1_metadata_access_permitted"] = False
blind_continuity_expected["a1_image_access_permitted"] = False
blind_continuity_expected["a1_label_access_permitted"] = False
blind_continuity_expected["a1_development_role_permitted"] = False
blind_continuity_expected["a1_reserve_substitute_permitted"] = False
blind_continuity_expected["blind_access_stage"] = "ONLY_AFTER_FINAL_MODEL_STATE_AND_LABEL_FREE_PREDICTIONS_ARE_FROZEN_UNDER_EXISTING_FUTURE_BLIND_PROTOCOL"

if REPLAY_MODE:
    blind_continuity = pd.read_csv(BLIND_CONTINUITY_PATH)
else:
    blind_continuity = blind_continuity_expected.copy()
    write_immutable_csv(BLIND_CONTINUITY_PATH, blind_continuity)
assert canonical_csv_text(blind_continuity) == canonical_csv_text(blind_continuity_expected)

AMENDED_STAGE11C_AUTHORISED = {
    "original_official_route_recovery": True,
    "manual_official_download_receipt_adjudication": True,
    "reserve_metadata_rights_schema_screening_in_frozen_order": True,
    "reserve_asset_acquisition_only_after_pre_activation_metadata_gate": True,
    "development_roster_seal_before_embeddings": True,
    "source_axis_fit_after_roster_and_governance_seal": True,
    "development_transfer_edges_after_source_gate": True,
    "locked_blind_metadata_or_asset_access": False,
    "DDO2_fit": False,
    "Stage12": False,
    "blind_validation": False,
}
assert not any(AMENDED_STAGE11C_AUTHORISED[key] for key in ["locked_blind_metadata_or_asset_access", "DDO2_fit", "Stage12", "blind_validation"])
print("Locked-blind roles unchanged:", blind_continuity["dataset_id"].tolist())
print("Amended Stage11C may recover/screen/seal/source-gate development only; blind access:", AMENDED_STAGE11C_AUTHORISED["locked_blind_metadata_or_asset_access"])

Locked-blind roles unchanged: ['BUSI_CAIRO_2019', 'OASBUD_2017', 'DERM7PT_2019']
Amended Stage11C may recover/screen/seal/source-gate development only; blind access: False


In [15]:
# @title A1-5. Independently validate lineage, outcome-free selection, capacity, and blind isolation
validity_rows = []
def check(name, passed, evidence, severity="BLOCKER"):
    validity_rows.append({"check": name, "passed": bool(passed), "severity_if_failed": severity, "evidence": str(evidence)})

check("stage9_final_self_hash", stage9_final["final_record_sha256"] == EXPECTED_STAGE9_FINAL_HASH, stage9_final["final_record_sha256"])
check("stage10_final_self_hash", stage10_final["final_record_sha256"] == EXPECTED_STAGE10_FINAL_HASH, stage10_final["final_record_sha256"])
check("stage10_protocol_continuity", stage10_final["stage10_protocol_seal_sha256"] == EXPECTED_STAGE10_PROTOCOL_HASH, stage10_final["stage10_protocol_seal_sha256"])
check("stage10_role_registry_hash", sha256_file(STAGE10_ROLE_PATH) == EXPECTED_ROLE_REGISTRY_HASH, sha256_file(STAGE10_ROLE_PATH))
check("stage11b_final_self_hash", stage11b_final["final_record_sha256"] == EXPECTED_STAGE11B_FINAL_HASH, stage11b_final["final_record_sha256"])
check("stage11b_no_repair_ready", not stage11b_final["stage11c_authorised_for_exact_handoff_only"] and not stage11b_final["stage11c_handoff_datasets"], stage11b_final["stage11c_handoff_datasets"])
check("stage11b_no_performance_failure_claim", not stage11b_final["source_recoverability_evaluated"], stage11b_final["source_recoverability_evaluated"])
check("five_originals_exact_and_ordered", route_registry["dataset_id"].tolist() == ORIGINAL_IDS, route_registry["dataset_id"].tolist())
check("original_roles_retained", route_registry["original_role_retained"].eq("DEVELOPMENT_EXTENSION").all(), route_registry["original_role_retained"].unique())
check("official_route_only", not route_registry["permitted_route_order"].str.contains("KAGGLE|HUGGINGFACE|MIRROR", case=False, regex=True).any(), route_registry["permitted_route_order"].tolist())
check("manual_route_requires_receipt", route_registry["permitted_route_order"].str.contains("MANUAL_OFFICIAL").all() and manual_receipt_schema.loc[manual_receipt_schema["field"].eq("computed_sha256"), "required"].astype(str).str.lower().eq("true").all(), "manual official route plus computed_sha256")
check("reserve_priority_unique_contiguous", reserve_pool["reserve_priority"].tolist() == list(range(1, len(reserve_pool) + 1)), reserve_pool["reserve_priority"].tolist())
check("reserves_not_accessed_or_activated", reserve_pool["status"].eq("CONDITIONAL_RESERVE_NOT_ACCESSED_NOT_ACTIVATED").all(), reserve_pool["status"].tolist())
check("reserve_roles_development_only", reserve_pool["role_if_activated"].eq("DEVELOPMENT_EXTENSION_A1_RESERVE").all(), reserve_pool["role_if_activated"].unique())
check("reserve_modality_group_preserved", reserve_pool["modality"].eq("breast_ultrasound").all(), reserve_pool["modality"].unique())
check("reserve_and_blind_ids_disjoint", set(reserve_pool["dataset_id"]).isdisjoint(LOCKED_BLIND_IDS), set(reserve_pool["dataset_id"]) & set(LOCKED_BLIND_IDS))
check("known_derivative_and_blind_mirrors_excluded", {"BUSC_2023", "BUSI_CAIRO_MIRRORS"}.issubset(set(exclusion_registry["dataset_id_or_family"])), exclusion_registry["dataset_id_or_family"].tolist())
check("composite_source_excluded", "BUS_COT_2026_COMPOSITE" in set(exclusion_registry["dataset_id_or_family"]), exclusion_registry["dataset_id_or_family"].tolist())
check("target_roster_is_five", activation_policy["target_development_domain_count"] == 5, activation_policy["target_development_domain_count"])
check("additional_edge_need_is_nine", activation_policy["additional_edges_needed"] == 9, activation_policy["additional_edges_needed"])
check("minimum_feasible_domain_count_is_four", activation_policy["minimum_qualified_domains_with_any_count_gate_possibility"] == 4, activation_policy["minimum_qualified_domains_with_any_count_gate_possibility"])
check("three_domains_mathematically_insufficient", int(capacity_table.loc[capacity_table["qualified_development_domains"].eq(3), "maximum_new_directed_edges_if_all_sources_recoverable"].iloc[0]) == 6, capacity_table.to_dict("records"))
check("four_domains_three_sources_exactly_sufficient", 3 * (4 - 1) == 9, 3 * (4 - 1))
check("activation_inputs_are_outcome_free", set(activation_policy["activation_inputs_permitted"]).isdisjoint(set(activation_policy["activation_inputs_prohibited"])), activation_policy["activation_inputs_permitted"])
check("no_post_performance_replacement", "never activates another reserve" in activation_policy["post_roster_rule"], activation_policy["post_roster_rule"])
check("locked_blind_count_and_order_unchanged", blind_continuity["dataset_id"].tolist() == LOCKED_BLIND_IDS, blind_continuity["dataset_id"].tolist())
check("locked_blind_all_access_false", not blind_continuity[["a1_metadata_access_permitted", "a1_image_access_permitted", "a1_label_access_permitted", "a1_development_role_permitted", "a1_reserve_substitute_permitted"]].astype(bool).any().any(), "all false")
stable_runtime_boundary = {key: runtime_state[key] for key in ["external_requests_made", "images_or_labels_accessed", "locked_blind_assets_touched", "models_fitted"]}
check("no_data_or_model_action_in_a1", not any(stable_runtime_boundary.values()), stable_runtime_boundary)
check("amended_stage11c_still_blocks_stage12_and_blind", not AMENDED_STAGE11C_AUTHORISED["Stage12"] and not AMENDED_STAGE11C_AUTHORISED["blind_validation"], AMENDED_STAGE11C_AUTHORISED)

validity = pd.DataFrame(validity_rows)
write_immutable_csv(VALIDITY_PATH, validity)
if not validity["passed"].all():
    display(validity.loc[~validity["passed"]])
    raise AssertionError("A1 validity gate failed; amendment must not be sealed.")
print("Independent blocker checks passed:", int(validity["passed"].sum()), "/", len(validity))

Independent blocker checks passed: 29 / 29


In [16]:
# @title A1-6. Seal the amendment protocol, report, integrity manifest, final record, and replay state
import matplotlib.pyplot as plt

ANALYSIS_SPEC = {
    "scope": "PROSPECTIVE_DEVELOPMENT_ACQUISITION_GOVERNANCE_AMENDMENT_ONLY",
    "amends": ["Stage10 no-alternative-route rule", "Stage10 no-substitution rule after upstream availability/governance failure"],
    "preserves": ["Stage9 21-edge evidence", "Stage9 model and validation specification", "Stage10 original development priorities", "three locked-blind roles", "thirteen candidate blind edges", "all Stage11/11A/11B audit records"],
    "official_route_rule": "current official API, official landing download, official client/tool, publisher or author official repository, or checksum-audited manual official download",
    "reserve_rule": "originals first; frozen reserves in rank order; activation uses pre-model acquisition/governance evidence only; final roster sealed before embedding or source fit",
    "capacity_rule": activation_policy["edge_formula"],
    "minimum_qualified_domains_for_count_gate": 4,
    "target_development_domains": 5,
    "prohibited": ["unofficial mirrors", "outcome-based substitution", "post-source-gate replacement", "locked-blind access", "DDO2 fit", "Stage12", "blind validation"],
}

protocol_payload = {
    "stage": "ProtocolAmendmentA1", "protocol_version": "0.1",
    "decision": "SEAL_OFFICIAL_ROUTE_RECOVERY_AND_OUTCOME_FREE_DEVELOPMENT_ONLY_SUBSTITUTION_RULES",
    "parent_stage9_final_record_sha256": EXPECTED_STAGE9_FINAL_HASH,
    "parent_stage10_final_record_sha256": EXPECTED_STAGE10_FINAL_HASH,
    "parent_stage10_protocol_seal_sha256": EXPECTED_STAGE10_PROTOCOL_HASH,
    "parent_role_registry_sha256": EXPECTED_ROLE_REGISTRY_HASH,
    "parent_stage11b_final_record_sha256": EXPECTED_STAGE11B_FINAL_HASH,
    "parent_stage11b_protocol_seal_sha256": EXPECTED_STAGE11B_PROTOCOL_HASH,
    "notebook_source_sha256": notebook_source_hash,
    "official_route_registry_sha256": sha256_file(ROUTE_REGISTRY_PATH),
    "manual_receipt_schema_sha256": sha256_file(MANUAL_RECEIPT_SCHEMA_PATH),
    "reserve_pool_sha256": sha256_file(RESERVE_POOL_PATH),
    "exclusion_registry_sha256": sha256_file(EXCLUSION_REGISTRY_PATH),
    "activation_policy_sha256": activation_policy["policy_sha256"],
    "locked_blind_continuity_sha256": sha256_file(BLIND_CONTINUITY_PATH),
    "analysis_spec": ANALYSIS_SPEC,
}

if REPLAY_MODE:
    protocol_seal = json.loads(PROTOCOL_SEAL_PATH.read_text(encoding="utf-8"))
    seal_claim = protocol_seal["seal_sha256"]
    without_claim = dict(protocol_seal); without_claim.pop("seal_sha256")
    assert sha256_json(without_claim) == seal_claim
    for key, value in protocol_payload.items():
        assert protocol_seal[key] == value
else:
    protocol_seal = dict(protocol_payload)
    protocol_seal["sealed_utc"] = utc_now()
    protocol_seal["seal_sha256"] = sha256_json(protocol_seal)
    write_immutable_json(PROTOCOL_SEAL_PATH, protocol_seal)
    seal_claim = protocol_seal["seal_sha256"]

if not REPLAY_MODE:
    x = capacity_table["qualified_development_domains"].to_numpy()
    max_edges = capacity_table["maximum_new_directed_edges_if_all_sources_recoverable"].to_numpy()
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    axes[0].bar(x, max_edges, color=["#adb5bd", "#adb5bd", "#e9c46a", "#2a9d8f", "#2a9d8f"])
    axes[0].axhline(ADDITIONAL_EDGES_NEEDED, color="#c1121f", linestyle="--", label="9 new edges needed")
    axes[0].set_xlabel("Acquisition/governance-qualified domains")
    axes[0].set_ylabel("Maximum new directed edges")
    axes[0].set_title("Why three domains are insufficient")
    axes[0].legend(frameon=False)
    required_sources = [np.nan if pd.isna(v) else v for v in capacity_table["minimum_recoverable_sources_for_nine_new_edges"]]
    axes[1].plot(x, required_sources, marker="o", color="#264653", linewidth=2)
    axes[1].plot(x, x, linestyle="--", color="#8d99ae", label="all domains recoverable")
    axes[1].set_xlabel("Qualified domains")
    axes[1].set_ylabel("Minimum recoverable sources")
    axes[1].set_title("Count-gate feasibility")
    axes[1].set_xticks(x)
    axes[1].legend(frameon=False)
    fig.suptitle("Protocol Amendment A1 — development capacity before any performance inspection", fontsize=12)
    fig.tight_layout()
    fig.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight", metadata={"Software": "ProtocolAmendmentA1"})
    plt.close(fig)

    report = f"""# Protocol Amendment A1 — Official-Route Recovery and Development-Only Substitution Rules

## Decision

`SEAL_OFFICIAL_ROUTE_RECOVERY_AND_OUTCOME_FREE_DEVELOPMENT_ONLY_SUBSTITUTION_RULES`

Stage 11B ended with zero repair-ready development domains under the old route restrictions, but it did not observe a source-recoverability performance failure. A1 therefore amends acquisition governance without changing any scientific outcome or blind role.

## Frozen recovery and substitution logic

- The five Stage 10 development datasets remain first priority in their original order.
- Alternative access is allowed only through current official APIs, official landing-page downloads, official clients/tools, publisher or author repositories, or a manual official download with a complete checksum receipt.
- Three conditional reserves are frozen in order: `{', '.join(reserve_pool['dataset_id'])}`. None is accessed or activated by A1.
- Every reserve must pass rights, pathology-compatible label, patient/lesion grouping, integrity, and duplicate/provenance gates before activation.
- Known derivatives, BUSI mirrors, and mixed-source composites are explicitly excluded.
- The final development roster is sealed before embeddings or source fitting. A poor source or transfer result can never trigger a replacement.

## Correct capacity boundary

Stage 9 has 21 eligible edges and requires at least 30, so 9 new edges are needed. With `N` qualified development domains and `S` recoverable sources, the maximum eligible directed edges contributed by this modality are `S × (N−1)`. Three domains can produce at most 6 edges and are insufficient. Four domains with three recoverable sources can produce exactly 9; the target remains five domains.

## Unchanged blind boundary

`{', '.join(LOCKED_BLIND_IDS)}` and all 13 candidate blind edges remain unchanged and inaccessible. A1 performs no network request, image/label access, embedding, model fit, DDO2 fit, Stage 12 action, or blind validation.

## Next step

`BUILD_AMENDED_STAGE11C_ORIGINAL_FIRST_OFFICIAL_ROUTE_RECOVERY_AND_OUTCOME_FREE_RESERVE_SCREENING`
"""
    write_immutable_text(REPORT_PATH, report)

tracked_paths = [
    PROTOCOL_SEAL_PATH, PARENT_COMMITMENT_PATH, ROUTE_REGISTRY_PATH, MANUAL_RECEIPT_SCHEMA_PATH,
    RESERVE_POOL_PATH, EXCLUSION_REGISTRY_PATH, ACTIVATION_POLICY_PATH, BLIND_CONTINUITY_PATH,
    VALIDITY_PATH, REPORT_PATH, FIGURE_PATH,
]

if not REPLAY_MODE:
    output_manifest = pd.DataFrame([{
        "relative_path": str(path.relative_to(A1_ROOT)), "size_bytes": path.stat().st_size, "sha256": sha256_file(path)
    } for path in tracked_paths])
    write_immutable_csv(OUTPUT_MANIFEST_PATH, output_manifest)
else:
    output_manifest = pd.read_csv(OUTPUT_MANIFEST_PATH)

for row in output_manifest.itertuples(index=False):
    path = A1_ROOT / row.relative_path
    assert path.is_file() and path.stat().st_size == int(row.size_bytes) and sha256_file(path) == row.sha256

decision = "SEAL_A1_AUTHORISE_AMENDED_STAGE11C_DEVELOPMENT_RECOVERY_AND_PREFIT_RESERVE_SCREENING_KEEP_STAGE12_DDO2_AND_BLIND_VALIDATION_PROHIBITED"
if REPLAY_MODE:
    final_record = verify_self_hashed_json(FINAL_RECORD_PATH, json.loads(FINAL_RECORD_PATH.read_text(encoding="utf-8"))["final_record_sha256"])
    final_claim = final_record["final_record_sha256"]
else:
    tracked_bytes = sum(path.stat().st_size for path in tracked_paths + [OUTPUT_MANIFEST_PATH])
    assert tracked_bytes <= MAXIMUM_NEW_A1_BYTES
    final_record = {
        "stage": "ProtocolAmendmentA1", "decision": decision,
        "scope": "DEVELOPMENT_ACQUISITION_GOVERNANCE_AMENDMENT_ONLY_NO_DATA_ACCESS_NO_MODEL_NO_BLIND_ACTION",
        "parent_stage9_final_record_sha256": EXPECTED_STAGE9_FINAL_HASH,
        "parent_stage10_final_record_sha256": EXPECTED_STAGE10_FINAL_HASH,
        "parent_stage10_protocol_seal_sha256": EXPECTED_STAGE10_PROTOCOL_HASH,
        "parent_role_registry_sha256": EXPECTED_ROLE_REGISTRY_HASH,
        "parent_stage11b_final_record_sha256": EXPECTED_STAGE11B_FINAL_HASH,
        "parent_stage11b_protocol_seal_sha256": EXPECTED_STAGE11B_PROTOCOL_HASH,
        "a1_protocol_seal_sha256": seal_claim,
        "notebook_source_sha256": notebook_source_hash,
        "original_development_datasets_retained": len(route_registry),
        "conditional_reserve_datasets_frozen": len(reserve_pool),
        "explicit_exclusions_frozen": len(exclusion_registry),
        "locked_blind_datasets_unchanged": len(blind_continuity),
        "candidate_locked_blind_edges_unchanged": int(stage10_final["candidate_locked_blind_edges"]),
        "current_eligible_edges": CURRENT_ELIGIBLE_EDGES,
        "additional_edges_needed": ADDITIONAL_EDGES_NEEDED,
        "target_development_domain_count": TARGET_DEVELOPMENT_DOMAIN_COUNT,
        "minimum_qualified_domains_for_any_count_gate_possibility": MINIMUM_QUALIFIED_DOMAINS_FOR_COUNT_GATE,
        "official_route_registry_sha256": sha256_file(ROUTE_REGISTRY_PATH),
        "manual_receipt_schema_sha256": sha256_file(MANUAL_RECEIPT_SCHEMA_PATH),
        "reserve_pool_sha256": sha256_file(RESERVE_POOL_PATH),
        "exclusion_registry_sha256": sha256_file(EXCLUSION_REGISTRY_PATH),
        "activation_policy_sha256": activation_policy["policy_sha256"],
        "locked_blind_continuity_sha256": sha256_file(BLIND_CONTINUITY_PATH),
        "independent_validity_checks_sha256": sha256_file(VALIDITY_PATH),
        "output_integrity_manifest_sha256": sha256_file(OUTPUT_MANIFEST_PATH),
        "tracked_a1_output_bytes_excluding_runtime_and_final_record": tracked_bytes,
        "maximum_new_a1_bytes": MAXIMUM_NEW_A1_BYTES,
        "external_requests_made": False, "images_accessed": False, "labels_accessed": False,
        "reserves_accessed": False, "reserves_activated": False, "embeddings_computed": False,
        "source_axes_fitted": False, "transfer_edges_evaluated": False, "ddo2_coefficients_fitted": False,
        "stage12_authorised": False, "locked_blind_assets_touched": False, "blind_validation_performed": False,
        "next_step": "BUILD_AMENDED_STAGE11C_ORIGINAL_FIRST_OFFICIAL_ROUTE_RECOVERY_AND_OUTCOME_FREE_RESERVE_SCREENING",
        "completed_utc": utc_now(),
    }
    final_record["final_record_sha256"] = sha256_json(final_record)
    final_claim = final_record["final_record_sha256"]
    write_immutable_json(FINAL_RECORD_PATH, final_record)
    runtime_state.update({"completed": True, "decision": decision, "protocol_seal_sha256": seal_claim, "final_record_sha256": final_claim, "last_updated_utc": utc_now()})
    atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("\n================ PROTOCOL AMENDMENT A1 COMPLETE ================")
print("Original development datasets retained:", len(route_registry))
print("Conditional reserves / explicit exclusions:", len(reserve_pool), "/", len(exclusion_registry))
print("Locked-blind datasets unchanged:", len(blind_continuity))
print("Target roster / minimum feasible domains:", TARGET_DEVELOPMENT_DOMAIN_COUNT, "/", MINIMUM_QUALIFIED_DOMAINS_FOR_COUNT_GATE)
print("Decision:", final_record["decision"])
print("Protocol seal:", seal_claim)
print("Official route registry hash:", sha256_file(ROUTE_REGISTRY_PATH))
print("Reserve pool hash:", sha256_file(RESERVE_POOL_PATH))
print("Activation policy hash:", activation_policy["policy_sha256"])
print("Final record hash:", final_claim)
print("Next step:", final_record["next_step"])


================ PROTOCOL AMENDMENT A1 COMPLETE ================
Original development datasets retained: 5
Conditional reserves / explicit exclusions: 3 / 4
Locked-blind datasets unchanged: 3
Target roster / minimum feasible domains: 5 / 4
Decision: SEAL_A1_AUTHORISE_AMENDED_STAGE11C_DEVELOPMENT_RECOVERY_AND_PREFIT_RESERVE_SCREENING_KEEP_STAGE12_DDO2_AND_BLIND_VALIDATION_PROHIBITED
Protocol seal: bb85f49c2378600a50619624ad322dc824d662d8eadb73fcf1d5532ee067082a
Official route registry hash: baecfc013349037d739b5058e45938e0f785718465bd88d45d58de5f3f4b7399
Reserve pool hash: 37c1d5da4cfd2057ee3438cacfbc831df6d4cdcb5c7efd87cf629d8cf871e912
Activation policy hash: fd9dc25be5699c4dee17dc7af18f1bb9e063bafc6d7835087212e113ead0ce6a
Final record hash: 4003be92df8a9c84352062383e8a94d066cc0e2bb2978cfb1af8968e51b85320
Next step: BUILD_AMENDED_STAGE11C_ORIGINAL_FIRST_OFFICIAL_ROUTE_RECOVERY_AND_OUTCOME_FREE_RESERVE_SCREENING
